In [7]:
import pandas as pd
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [2]:
df_answers = pd.read_csv("data/rag_answers.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
# Instructions for the AQA judge.This tells the judge what to compare and how to assign the score

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [9]:
rec = answers[0]

In [10]:
rec

{'question': 'I just found this course — is it too late to join now?',
 'answer_llm': 'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [11]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [ ]:
# Evaluate the first answer using the AQA judge

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says it is still possible to join, and that receiving a certificate requires submitting the project while submissions are still accepted/open. This is semantically equivalent.', score='good')

In [ ]:
# judgement of the judge on the first answer

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: it says it is still possible to join, and that receiving a certificate requires submitting the project while submissions are still accepted/open. This is semantically equivalent.', score='good')

In [14]:
calc_price(usage)

{'input_cost': 0.00021825000000000002,
 'output_cost': 0.0002655,
 'total_cost': 0.00048375}

In [ ]:
# define a function to evaluate a question and answer pair using the AQA judge

def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [16]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it says the course can still be joined now, and that a certificate requires submitting the project while submissions are open. This is semantically equivalent.', score='good')

In [17]:
# define a function to judge a record

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [18]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [20]:
results[:5]

[({'question': 'I just found this course — is it too late to join now?',
   'document': '74eb249bbf',
   'score': 'good',
   'reasoning': 'The AI answer preserves the key meaning of the ground truth: it is not too late to join, but certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.'},
  ResponseUsage(input_tokens=291, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=54, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=345)),
 ({'question': 'Can I still take the course if I discovered it late?',
   'document': '74eb249bbf',
   'score': 'good',
   'reasoning': 'The AI answer preserves the key point of the original: late discovery does not prevent joining, but certificate eligibility requires submitting the project before submissions close. This is semantically equivalent.'},
  ResponseUsage(input_tokens=296, input_tokens_details=InputTokensDetails(cach

In [21]:
# split the results into two lists: evaluations and usages

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [22]:
df_eval = pd.DataFrame(evaluations)

In [23]:
df_eval.head()

,question,document,score,reasoning
0,I just found this course — is it too late to j...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still take the course if I discovered it...,74eb249bbf,good,The AI answer preserves the key point of the o...
2,Is it okay to join the course after it already...,74eb249bbf,good,The AI answer preserves the key points: joinin...
3,"If I join now, can I still get a certificate s...",74eb249bbf,good,The AI answer preserves the key point: joining...
4,What do I need to do to qualify for the certif...,74eb249bbf,good,The ground truth says late joiners can still g...


In [ ]:
# calculating total price of the API calls made during the evaluation

calc_total_price(usages)

0.40030275

In [25]:
# Calculate the percentage of "good" answers

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 545/565 = 96.46%


In [26]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
28,Is peer reviewing the capstone project require...,69d122f12e,bad,The AI answer incorrectly says 'Yes' to the qu...
30,Do I still qualify for the certificate if I sk...,9f689c185f,bad,The AI answer preserves the main point that sk...
31,Is homework required to get the course certifi...,9f689c185f,bad,The AI answer captures the main point that hom...
32,What do I need to pass in order to receive the...,9f689c185f,bad,The ground truth says the certificate requires...
83,Will skipping a homework hurt my chance to get...,cdc3b285e5,bad,The AI answer correctly states that skipping h...


In [27]:
df_eval.to_csv("data/rag_evaluations_results.csv", index=False)